In [ ]:
dbutils.widgets.text("catalog", "")
catalog = dbutils.widgets.get("catalog")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.config")

spark.sql(f"DROP TABLE IF EXISTS {catalog}.config.data_quality_rules")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.config.data_quality_rules (
        domain STRING,
        type STRING,
        column STRING,
        rule STRING
    )
    USING DELTA
""")

def interpolation_rows(domain, columns):
    return ",\n".join(
        f"""('{domain}', 'case', '{c}',
            'CASE WHEN {c} IS NULL THEN (LAST({c}, true) OVER (ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) + FIRST({c}, true) OVER (ORDER BY date ROWS BETWEEN 1 FOLLOWING AND UNBOUNDED FOLLOWING)) / 2 ELSE {c} END')"""
        for c in columns
    )

yfinance_columns = ["open", "high", "low", "close", "adj_close", "volume"]
fred_columns = ["GDPC1", "CPIAUCSL", "UNRATE", "FEDFUNDS", "DFF", "DGS10", "DGS2", "T10Y2Y", "VIXCLS", "M2SL"]

rows = ",\n".join([interpolation_rows("yfinance", yfinance_columns), interpolation_rows("fred", fred_columns)])

spark.sql(f"INSERT INTO {catalog}.config.data_quality_rules VALUES {rows}")